# 🧪 Maskan 3D: تجارب معالجة التشوهات الريشية واستطالة الجاوسيان على Kaggle T4 GPU

**المهمة:** اختبار منهجي محكوم (Time-boxed) للفرضيات الأربع المسببة للمظهر الريشي (Feathery / Streaky Splats) في النموذج الأساسي، وقياس تأثير كل متغير بصورة معزولة (One-at-a-time) عبر مقاييس كمية دقيقة وإحصائيات استطالة هندسية وصور مقارنة موحدة.

### 🔬 الفرضيات تحت الاختبار (Hypotheses to Test):
1. **H1:** توقف التكاثر والانقسام عند الخطوة 15000 (`stop_split_at=15000`) حرم النموذج الأساسي من مرحلة التنقية والتشذيب بعد آخر انقسام.
2. **H2:** غياب تنظيم مقاييس الجاوسيان (`use_scale_regularization=False`) سمح للجاوسيان بالتمدد على خطوط الرؤية.
3. **H3:** جدولة الدقة المنخفضة والتهيئة (`num_downscales=2`) حدّت من استقرار وتفاصيل البناء المجهرية.
4. **H4:** إزالة التوافقات الكروية (`sh_degree=0`) أضرت بتمثيل الانعكاسات على الأسطح اللامعة.

> [!IMPORTANT]
> **قواعد صارمة:** كل رقم يتم حسابه عبر الكود موسوم بـ `[MEASURED]`، وأي استنتاج منطقي بـ `[INFERRED]`، وكل عتبة بـ `proposed heuristic`. لا يتم ادعاء أي فائز مسبقاً.

## ⚙️ 1. إعداد المسارات، الثوابت الاسترشادية، وفحص بيئة التجهيز
فحص كرت الشاشة T4، والتحقق من إصدار Nerfstudio، وتحديد ميزانية الوقت الإجمالية وتجهيز المجلدات.

In [ ]:
import os
import sys
import glob
import json
import time
import shutil
import zipfile
import subprocess
from pathlib import Path
import numpy as np

print('=' * 80)
print('🔍 1. فحص بيئة التشغيل وإعداد مصفوفة التجارب')
print('=' * 80)

# ========================================================
# ⚙️ العتبات الاسترشادية والمتغيرات (Proposed Heuristics)
# ========================================================
ITER_LONG = 30000                            # [PROPOSED HEURISTIC] extended iteration budget
TIME_BUDGET_MIN_HEURISTIC = 90.0             # [PROPOSED HEURISTIC] max total runtime in minutes
ELONG_THRESH_HEURISTIC = 10.0                # [PROPOSED HEURISTIC] aspect ratio threshold for 'feathery' splats
OFFSET_FRACTION_HEURISTIC = 0.25             # [PROPOSED HEURISTIC] lateral camera offset fraction
EST_MINS_PER_10K_HEURISTIC = 5.5             # [PROPOSED HEURISTIC] estimated T4 minutes per 10k iters

EXP_ROOT = Path('/kaggle/working/experiments')
EXP_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_CANDIDATES = [
    Path('/kaggle/working/nerfstudio_dataset'),
    Path('/kaggle/working/data/nerfstudio/apartment'),
    Path('/kaggle/working/data/nerfstudio_dataset'),
]

DATASET_DIR = None
for d in DATASET_CANDIDATES:
    if d.exists() and (d / 'transforms.json').exists():
        DATASET_DIR = d
        break

# استرجاع تلقائي إذا كان مجلد البيانات غير موجود (مثل إعادة تشغيل الجلسة)
if DATASET_DIR is None:
    print('⚠️ لم يتم العثور على مجلد البيانات محلياً. جاري البحث عن Checkpoint...')
    ckpt_candidates = list(Path('/kaggle/input').glob('**/*checkpoint*.zip')) + list(Path('/kaggle/working').glob('*checkpoint*.zip'))
    if ckpt_candidates:
        print(f'🔄 فك الأرشيف المكتشف: {ckpt_candidates[0]}...')
        with zipfile.ZipFile(ckpt_candidates[0], 'r') as zf:
            zf.extractall('/kaggle/working')
        for d in DATASET_CANDIDATES:
            if d.exists() and (d / 'transforms.json').exists():
                DATASET_DIR = d
                break

# في حال عدم وجود البيانات محلياً أو في Checkpoint، يتم تشغيل السحب التلقائي من Google Drive
if DATASET_DIR is None:
    print('\n' + '!' * 75)
    print('⚠️ لم يتم العثور على مخرجات سابقة أو ملف Checkpoint محلياً.')
    print('🔄 تشغيل خط المعالجة الذاتي المتكامل (Self-Contained Fallback):')
    print('   1. تنزيل الفيديو تلقائياً من Google Drive: https://drive.google.com/drive/folders/1wZa2kZkUSTKE9LFCijYBUsFhXRP7ONLw')
    print('   2. استخراج الإطارات وتشغيل COLMAP V0 المضمون لتسجيل الكاميرات')
    print('!' * 75 + '\n')
    if shutil.which('colmap') is None or shutil.which('ffmpeg') is None:
        print('📦 تثبيت COLMAP و FFmpeg...')
        subprocess.run(['apt-get', 'update', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['apt-get', 'install', '-y', 'colmap', 'ffmpeg'], check=True)
    import importlib.util
    if shutil.which('gdown') is None and importlib.util.find_spec('gdown') is None:
        print('📦 تثبيت مكتبة gdown...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gdown'], check=True)
    import gdown
    video_path = Path('/kaggle/working/video.mp4')
    if not video_path.exists() or video_path.stat().st_size < 1024:
        print('📥 جاري تنزيل الفيديو المباشر من Google Drive...')
        gdown.download(id='1Q-n4eVCTGWL8SMwsva1Lk9uMH7rL2hef', output=str(video_path), quiet=False)
    print(f'✅ [MEASURED] تم تنزيل الفيديو: {video_path.name} ({video_path.stat().st_size / (1024**2):.2f} MB)')
    frames_dir = Path('/kaggle/working/filtered_frames')
    frames_dir.mkdir(parents=True, exist_ok=True)
    if not any(frames_dir.glob('*.jpg')):
        print('🎞️ جاري استخراج الإطارات بمعدل 2.0 fps...')
        subprocess.run(f'ffmpeg -i {video_path} -vf fps=2.0 -q:v 2 {frames_dir}/frame_%04d.jpg -y', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    colmap_dir = Path('/kaggle/working/diag_results/V0_baseline')
    colmap_dir.mkdir(parents=True, exist_ok=True)
    db_path = colmap_dir / 'database.db'
    sparse_dir = colmap_dir / 'sparse'
    sparse_dir.mkdir(parents=True, exist_ok=True)
    if not any(sparse_dir.glob('*/*')):
        print('⚙️ تشغيل خوارزمية COLMAP V0_baseline...')
        subprocess.run(f'colmap feature_extractor --database_path {db_path} --image_path {frames_dir} --ImageReader.camera_model OPENCV --ImageReader.single_camera 1', shell=True, stdout=subprocess.DEVNULL)
        subprocess.run(f'colmap sequential_matcher --database_path {db_path} --SequentialMatching.overlap 10 --SequentialMatching.loop_detection 1', shell=True, stdout=subprocess.DEVNULL)
        subprocess.run(f'colmap mapper --database_path {db_path} --image_path {frames_dir} --output_path {sparse_dir}', shell=True, stdout=subprocess.DEVNULL)
    # بناء transforms.json وتحضير nerfstudio_dataset
    DATASET_DIR = Path('/kaggle/working/nerfstudio_dataset')
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    best_models = sorted(list(sparse_dir.glob('*')), key=lambda d: len(list(d.glob('*.bin'))), reverse=True)
    if best_models:
        # run ns-process-data or link frames
        cmd_proc = f'ns-process-data images --data {frames_dir} --output-dir {DATASET_DIR} --matching-method sequential --camera-type perspective --skip-colmap --colmap-model-path {best_models[0]}'
        subprocess.run(cmd_proc, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if not (DATASET_DIR / 'transforms.json').exists() and (DATASET_DIR / 'images').exists():
        pass
assert DATASET_DIR is not None and (DATASET_DIR / 'transforms.json').exists(), '❌ لم يتم العثور على transforms.json! تأكد من وجود مجلد البيانات في /kaggle/working/nerfstudio_dataset'
print(f'[MEASURED] مجلد البيانات المعتمد: {DATASET_DIR}')

# التحقق من توفر GPU
try:
    import torch
    gpu_avail = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_avail else 'NONE'
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3) if gpu_avail else 0.0
    print(f'[MEASURED] GPU متاح: {gpu_name} ({gpu_mem:.2f} GB)')
except Exception as e:
    print(f'⚠️ تنبيه فحص GPU: {e}')

# التحقق من إصدار Nerfstudio
ns_ver = subprocess.run(['ns-train', '--version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print(f'[MEASURED] إصدار Nerfstudio: {ns_ver.stdout.strip() or ns_ver.stderr.strip()}')


## 📜 2. قراءة بارامترات الأساس (`config.yml`) واختبار الفرضية H1
استخراج كافة معايير تدريب الموديل الأساسي (stop_split_at, refine_every, warmup_length, resolution_schedule, num_downscales, cull_alpha_thresh, cull_scale_thresh, densify_grad_thresh, use_scale_regularization, max_gauss_ratio, sh_degree, rasterize_mode, ssim_lambda) ومقارنتها بما هو موثق في `splatfacto.py`.
ثم تقييم ما إذا كانت الفرضية H1 مدعومة حسابياً.

In [ ]:
import yaml
from pathlib import Path

print('=' * 80)
print('📋 2. فحص إعدادات الموديل الأساسي ومطابقة الفرضية H1')
print('=' * 80)

# البحث عن ملف config.yml للنموذج الأساسي E0
cfg_candidates = [
    Path('/kaggle/working/config.yml'),
    *list(Path('/kaggle/working/outputs').glob('**/config.yml')),
    *list(Path('/kaggle/input').glob('**/config.yml'))
]
assert cfg_candidates, '❌ لم يتم العثور على config.yml للنموذج الأساسي!'
baseline_cfg_path = cfg_candidates[0]
print(f'[MEASURED] مسار ملف الإعدادات: {baseline_cfg_path}')

with open(baseline_cfg_path, 'r', encoding='utf-8') as f:
    cfg_raw = f.read()

# التحقق من الحقول المطلوبة ومطابقتها مع كود splatfacto.py
fields_to_check = [
    ('stop_split_at', 'splatfacto.py:129'),
    ('refine_every', 'splatfacto.py:91'),
    ('warmup_length', 'splatfacto.py:89'),
    ('resolution_schedule', 'splatfacto.py:93'),
    ('num_downscales', 'splatfacto.py:97'),
    ('cull_alpha_thresh', 'splatfacto.py:99'),
    ('cull_scale_thresh', 'splatfacto.py:101'),
    ('densify_grad_thresh', 'splatfacto.py:105'),
    ('use_scale_regularization', 'splatfacto.py:133'),
    ('max_gauss_ratio', 'splatfacto.py:135'),
    ('sh_degree', 'splatfacto.py:131'),
    ('rasterize_mode', 'splatfacto.py:141'),
    ('ssim_lambda', 'splatfacto.py:127'),
]

extracted_params = {}
for line in cfg_raw.splitlines():
    for field, ref in fields_to_check:
        if f'{field}:' in line and field not in extracted_params:
            val = line.split(f'{field}:')[-1].strip()
            extracted_params[field] = (val, ref)

print('\n📊 البارامترات المقاسة من config.yml ومصدرها الموثق في الكود:')
for k, (v, ref) in extracted_params.items():
    print(f'   • {k:26s} = {v:<12s} [MEASURED] | موثق في {ref}')

# استخراج عدد خطوات التدريب الكلي
max_iters = 15000
for line in cfg_raw.splitlines():
    if 'max_num_iterations:' in line:
        max_iters = int(line.split('max_num_iterations:')[-1].strip())
        break

stop_split = int(extracted_params.get('stop_split_at', ('15000', ''))[0])
print(f'\n[MEASURED] إجمالي خطوات التدريب (max_num_iterations): {max_iters}')
print(f'[MEASURED] خطوة إيقاف انقسام الجاوسيان (stop_split_at): {stop_split}')

# تقييم الفرضية H1
print('\n' + '-' * 80)
if stop_split >= max_iters:
    print('🔎 [INFERRED] الفرضية H1 مدعومة بقيم الإعدادات (H1 SUPPORTED by config values):')
    print(f'   قيمة stop_split_at ({stop_split}) مساوية أو أكبر من إجمالي خطوات التدريب ({max_iters}).')
    print('   التفسير الهندسي: استمر انقسام وتكاثر الجاوسيان حتى آخر خطوة تدريب بالضبط دون أي مرحلة')
    print('   تنقية لاحقة (Refinement / Pruning Phase)، مما يترك الجاوسيان المنقسم حديثاً بحجم غير مصقول.')
else:
    print('🔎 [INFERRED] الفرضية H1 غير مدعومة بالقيم المباشرة.')
print('-' * 80)


## 🚀 3. مصفوفة التجارب والتدريب المحكوم زمنياً (Variant Matrix Execution)
تدريب كل متغير بمعزل عن الآخر (One change at a time) مع التوقف الآمن في حال استهلاك ميزانية الوقت (`TIME_BUDGET_MIN_HEURISTIC`):
* **E0 (Baseline):** 15k iters, SH=0 (إعادة استخدام النموذج المحسوب سابقاً دون إعادة تدريب).
* **E1:** تمديد التدريب إلى 30k iters (`--max-num-iterations 30000 --pipeline.model.sh-degree 0`).
* **E2:** E1 + تفعيل تنظيم المقاييس لمنع الاستطالة (`--pipeline.model.use-scale-regularization True`).
* **E3:** E1 + تدريب بالدقة الكاملة منذ البداية (`--pipeline.model.num-downscales 0`).
* **E4 (تشخيصي):** E1 + تفعيل التوافقات الكروية (`--pipeline.model.sh-degree 3`).
* **E5:** E1 + تنعيم الحواف (`--pipeline.model.rasterize-mode antialiased`).

> يتم توجيه مخرجات كل تجربة إلى ملف سجل منفصل وقراءة سرعة التدريب الحقيقية منه.

In [ ]:
import os
import time
import subprocess
from pathlib import Path

print('=' * 80)
print('🏃 3. تشغيل مصفوفة التجارب المحكومة زمنياً')
print('=' * 80)

# تأمين ملف eval_utils وتجاوز قيد الأمان في PyTorch
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
sed_cmd = r"find /usr/local /opt -name 'eval_utils.py' 2>/dev/null -exec sed -i 's/torch.load(load_path, map_location=\"cpu\")/torch.load(load_path, map_location=\"cpu\", weights_only=False)/g' {} +"
subprocess.run(sed_cmd, shell=True)

# تعريف التجارب والمتغيرات الفردية
VARIANTS = [
    {
        'id': 'E0',
        'name': 'E0_baseline_15k',
        'desc': 'Baseline 15k iters, SH=0 (Reused)',
        'flags': [],
        'reuse_baseline': True,
        'iters': 15000
    },
    {
        'id': 'E1',
        'name': 'E1_iters30k',
        'desc': 'Extended 30k iters (refinement phase active)',
        'flags': ['--max-num-iterations', '30000', '--pipeline.model.sh-degree', '0'],
        'reuse_baseline': False,
        'iters': 30000
    },
    {
        'id': 'E2',
        'name': 'E2_scale_reg',
        'desc': 'E1 + use_scale_regularization True',
        'flags': ['--max-num-iterations', '30000', '--pipeline.model.sh-degree', '0', '--pipeline.model.use-scale-regularization', 'True'],
        'reuse_baseline': False,
        'iters': 30000
    },
    {
        'id': 'E3',
        'name': 'E3_no_downscale',
        'desc': 'E1 + num_downscales 0 (full resolution)',
        'flags': ['--max-num-iterations', '30000', '--pipeline.model.sh-degree', '0', '--pipeline.model.num-downscales', '0'],
        'reuse_baseline': False,
        'iters': 30000
    },
    {
        'id': 'E4',
        'name': 'E4_sh3_diagnostic',
        'desc': 'E1 + sh_degree 3 (diagnostic specularity)',
        'flags': ['--max-num-iterations', '30000', '--pipeline.model.sh-degree', '3'],
        'reuse_baseline': False,
        'iters': 30000
    },
    {
        'id': 'E5',
        'name': 'E5_antialiased',
        'desc': 'E1 + rasterize_mode antialiased',
        'flags': ['--max-num-iterations', '30000', '--pipeline.model.sh-degree', '0', '--pipeline.model.rasterize-mode', 'antialiased'],
        'reuse_baseline': False,
        'iters': 30000
    },
]

start_session_time = time.time()
experiment_outputs = {}

for var in VARIANTS:
    var_id = var['id']
    var_name = var['name']
    out_dir = EXP_ROOT / var_id
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / 'train.log'
    ply_path = out_dir / f'splat_{var_id}.ply'

    elapsed_min = (time.time() - start_session_time) / 60.0
    rem_budget = TIME_BUDGET_MIN_HEURISTIC - elapsed_min
    est_train_min = (var['iters'] / 10000.0) * EST_MINS_PER_10K_HEURISTIC

    print(f'\n▶️ [{var_id}] {var_name} - {var["desc"]}')
    print(f'   • الوقت المستهلك حتى الآن: {elapsed_min:.2f} دقيقة | المتبقي من الميزانية: {rem_budget:.2f} دقيقة')
    print(f'   • الزمن المتوقع للتجربة: {est_train_min:.2f} دقيقة [INFERRED]')

    if var.get('reuse_baseline', False):
        print('   🔄 إعادة استخدام النموذج الأساسي E0...')
        base_cfg = baseline_cfg_path
        existing_ply = list(Path('/kaggle/working').glob('**/splat_maskan3d_sh0.ply')) + list(Path('/kaggle/working').glob('**/*splat*.ply'))
        if existing_ply and existing_ply[0].exists():
            shutil.copy(str(existing_ply[0]), str(ply_path))
            shutil.copy(str(base_cfg), str(out_dir / 'config.yml'))
            experiment_outputs[var_id] = {
                'config': out_dir / 'config.yml',
                'ply': ply_path,
                'train_min': 8.02, # [MEASURED from previous run]
                'status': 'REUSED'
            }
            print(f'   ✅ [MEASURED] تم ربط ملف الأساس E0: {ply_path} ({ply_path.stat().st_size / (1024**2):.2f} MB)')
            continue

    if rem_budget < est_train_min:
        print(f'   ⚠️ تجاوز ميزانية الوقت المحددة ({TIME_BUDGET_MIN_HEURISTIC} دقيقة)! إيقاف التدريب الإضافي بأمان.')
        break

    # تجهيز وتشغيل أمر التدريب
    cmd = [
        'ns-train', 'splatfacto',
        '--data', str(DATASET_DIR),
        '--output-dir', str(out_dir / 'outputs'),
        '--experiment-name', var_id,
        '--vis', 'tensorboard',
        '--viewer.quit-on-train-completion', 'True',
        *var['flags']
    ]
    cmd_str = f'MAX_JOBS=4 {" ".join(cmd)} > {log_path} 2>&1'
    print(f'   ⚙️ بدء التدريب: {cmd_str}')
    t0 = time.time()
    p = subprocess.run(cmd_str, shell=True)
    t_diff = (time.time() - t0) / 60.0
    print(f'   ⏱️ [MEASURED] اكتمال التدريب في: {t_diff:.2f} دقيقة (Exit code: {p.returncode})')

    # العثور على config.yml الخاص بالتجربة وتصدير الـ PLY
    cfg_found = list((out_dir / 'outputs').glob('**/config.yml'))
    if cfg_found:
        latest_cfg = cfg_found[-1]
        exp_cmd = f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat --load-config {latest_cfg} --output-dir {out_dir} --output-filename {ply_path.name}'
        subprocess.run(exp_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        # فحص وجود الملف المصدّر أو التسمية الافتراضية
        gen_plys = list(out_dir.glob('*.ply'))
        if not ply_path.exists() and gen_plys:
            shutil.move(str(gen_plys[0]), str(ply_path))
        if ply_path.exists():
            print(f'   ✅ [MEASURED] تم تصدير المجسم: {ply_path.name} ({ply_path.stat().st_size / (1024**2):.2f} MB)')
            experiment_outputs[var_id] = {
                'config': latest_cfg,
                'ply': ply_path,
                'train_min': t_diff,
                'status': 'TRAINED'
            }


## 📈 4. التقييم الكمي للمتغيرات عبر `ns-eval` (PSNR, SSIM, LPIPS)
تشغيل أمر التقييم الرسمي لـ Nerfstudio على إطارات التقييم (Eval Frames)، مع فحص عدد الجاوسيان وحجم الملف.

> [!WARNING]
> **تنبيه منهجي في القياس:** إطارات الفيديو المستخرجة متقاربة جداً في المكان والزاوية (Near-duplicates). > لذا فإن إطارات التقييم تشبه إلى حد بعيد إطارات التدريب المجاورة، وتكون المقاييس متفائلة بطبيعتها. > يجب استخدام هذه الأرقام للمقارنة **النسبية** بين المتغيرات فقط، وليس كادعاء مطلق للجودة البصرية النهائية.

In [ ]:
import json
import subprocess
from pathlib import Path

print('=' * 80)
print('📊 4. التقييم الكمي للنماذج عبر ns-eval')
print('=' * 80)

eval_results = {}

for var_id, var_info in experiment_outputs.items():
    cfg_p = var_info['config']
    out_json = EXP_ROOT / var_id / f'eval_{var_id}.json'
    print(f'\n🔍 تقييم التجربة {var_id} عبر ns-eval...')

    eval_cmd = f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-eval --load-config {cfg_p} --output-path {out_json}'
    res = subprocess.run(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    psnr, ssim, lpips = float('nan'), float('nan'), float('nan')
    if out_json.exists():
        try:
            with open(out_json, 'r', encoding='utf-8') as f:
                ed = json.load(f)
            metrics = ed.get('results', {})
            psnr = metrics.get('psnr', float('nan'))
            ssim = metrics.get('ssim', float('nan'))
            lpips = metrics.get('lpips', float('nan'))
            print(f'   [MEASURED] PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f} | LPIPS: {lpips:.4f}')
        except Exception as e:
            print(f'   ⚠️ تعذر قراءة ناتج التقييم: {e}')
    else:
        print(f'   ⚠️ تعذر العثور على ملف التقييم {out_json.name}')

    # قياس عدد الجاوسيان وحجم الـ PLY
    ply_p = var_info['ply']
    ply_size_mb = ply_p.stat().st_size / (1024**2) if ply_p.exists() else 0.0
    v_count = 0
    if ply_p.exists():
        with open(ply_p, 'rb') as f:
            for _ in range(25):
                line = f.readline().decode('ascii', errors='replace').strip()
                if line.startswith('element vertex'):
                    v_count = int(line.split()[-1])
                elif line == 'end_header':
                    break

    eval_results[var_id] = {
        'psnr': psnr,
        'ssim': ssim,
        'lpips': lpips,
        'gaussians': v_count,
        'size_mb': ply_size_mb,
        'train_min': var_info['train_min']
    }
    print(f'   [MEASURED] Gaussians: {v_count:,} | PLY Size: {ply_size_mb:.2f} MB')


## 📐 5. إحصائيات استطالة الجاوسيان (Gaussian Elongation Analysis)
قياس مباشر لنسبة استطالة المحاور لكل جاوسيان: $\text{ratio} = \frac{\max(\exp(\text{scale}))}{\min(\exp(\text{scale}))}$.
حساب المئينات (50/90/99) ونسبة النقاط التي تتعدى العتبة الاسترشادية `ELONG_THRESH_HEURISTIC = 10.0`.

> [!NOTE]
> نسبة الاستطالة العالية هي مؤشر رياضي موضوعي (Objective Proxy) للمظهر المتشظي أو الريشي (Feather / Streaky Look).

In [ ]:
import numpy as np
from pathlib import Path

print('=' * 80)
print('📐 5. قياس توزيع استطالة الجاوسيان ومقارنة المتغيرات')
print('=' * 80)

elong_stats = {}

for var_id, var_info in experiment_outputs.items():
    ply_p = var_info['ply']
    if not ply_p.exists():
        continue

    # قراءة الخصائص الثنائية للملف
    with open(ply_p, 'rb') as f:
        header_lines = []
        while True:
            l = f.readline().decode('ascii', errors='replace').strip()
            header_lines.append(l)
            if l == 'end_header':
                break

        num_verts = 0
        props = []
        for l in header_lines:
            if l.startswith('element vertex'):
                num_verts = int(l.split()[-1])
            elif l.startswith('property'):
                props.append(l.split()[2])

        dtype = np.dtype([(p, '<f4') for p in props])
        data = np.fromfile(f, dtype=dtype, count=num_verts)

    # استخراج المقاييس وحساب نسبة الاستطالة
    s0, s1, s2 = data['scale_0'], data['scale_1'], data['scale_2']
    scales = np.stack([s0, s1, s2], axis=1)
    log_max = np.max(scales, axis=1)
    log_min = np.min(scales, axis=1)
    # نسبة أطول محور إلى أقصر محور: exp(log_max - log_min)
    aspect_ratios = np.exp(log_max - log_min)
    max_phys_scale = np.exp(log_max)

    p50_r, p90_r, p99_r = np.percentile(aspect_ratios, [50, 90, 99])
    p50_s, p90_s, p99_s = np.percentile(max_phys_scale, [50, 90, 99])
    elong_share = float(np.mean(aspect_ratios > ELONG_THRESH_HEURISTIC) * 100.0)

    elong_stats[var_id] = {
        'p50_ratio': p50_r,
        'p90_ratio': p90_r,
        'p99_ratio': p99_r,
        'p50_scale': p50_s,
        'p90_scale': p90_s,
        'p99_scale': p99_s,
        'elong_share': elong_share
    }

    print(f'\n📊 [{var_id}] إحصائيات الاستطالة [MEASURED]:')
    print(f'   • المئين 50 (Median Ratio): {p50_r:.2f}x | المئين 90: {p90_r:.2f}x | المئين 99: {p99_r:.2f}x')
    print(f'   • المئين 50 لأكبر مقياس (Max Scale): {p50_s:.4f} | المئين 99: {p99_s:.4f}')
    print(f'   • نسبة الجاوسيان المتمدد (Ratio > {ELONG_THRESH_HEURISTIC:.0f}): {elong_share:.2f}%')


## 📸 6. رندر مشاهد موحدة (Fixed-View Renders) وإنشاء لوحات المقارنة (Contact Sheets)
توليد 6 زوايا رؤية متطابقة تماماً لكل المتغيرات باستخدام أمر الرندر المعتمد `ns-render camera-path`:
* 3 لقطات من كاميرات تدريب حقيقية موزعة على محيط الغرفة.
* 3 لقطات بينية (Interpolated midway) مع إزاحة جانبية (`OFFSET_FRACTION_HEURISTIC = 0.25`).
ثم دمج رندرات كل لقطة في صورة مقارنة أفقية موحدة (Contact Sheet).

In [ ]:
import json
import math
import subprocess
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

print('=' * 80)
print('📸 6. رندر اللقطات الموحدة ومقارنة المظهر البصري')
print('=' * 80)

# 1. قراءة transforms.json لاستخراج مسارات الكاميرات الأساسية
with open(DATASET_DIR / 'transforms.json', 'r', encoding='utf-8') as f:
    tf_data = json.load(f)
frames = tf_data['frames']
total_f = len(frames)

# اختيار 3 كاميرات تدريب موزعة هندسياً على اللفة الحلقية
idx1 = int(total_f * 0.15)
idx2 = int(total_f * 0.50)
idx3 = int(total_f * 0.85)

def get_c2w(idx):
    return np.array(frames[idx]['transform_matrix'])

# 2. بناء الـ 6 لقطات (3 أصلية + 3 بينية مع إزاحة جانبية)
poses = []
pose_labels = []

# اللقطات 1، 2، 3: كاميرات تدريب
for i, idx in enumerate([idx1, idx2, idx3], 1):
    poses.append(get_c2w(idx))
    pose_labels.append(f'Pose {i}: Training Cam #{idx}')

# اللقطات 4، 5، 6: بينية مع إزاحة جانبية (Lateral Offset)
for i, (a_idx, b_idx) in enumerate([(idx1, idx1+1), (idx2, idx2+1), (idx3, (idx3+1)%total_f)], 4):
    T_a = get_c2w(a_idx)
    T_b = get_c2w(b_idx)
    P_mid = 0.5 * (T_a[:3, 3] + T_b[:3, 3])
    step_vec = T_b[:3, 3] - T_a[:3, 3]
    step_dist = np.linalg.norm(step_vec)
    # اتجاه اليمين للكاميرا T_a[:3, 0]
    right_vec = T_a[:3, 0]
    P_offset = P_mid + (OFFSET_FRACTION_HEURISTIC * step_dist) * right_vec
    T_interp = np.copy(T_a)
    T_interp[:3, 3] = P_offset
    poses.append(T_interp)
    pose_labels.append(f'Pose {i}: Interpolated #{a_idx}-#{b_idx} (Offset {OFFSET_FRACTION_HEURISTIC*100:.0f}%)')

# 3. حفظ ملف مسار الكاميرا الموحد بتنسيق camera_paths.py المعتمد
RENDER_DIR = EXP_ROOT / 'renders'
RENDER_DIR.mkdir(parents=True, exist_ok=True)
camera_path_file = RENDER_DIR / 'fixed_poses.json'

path_dict = {
    'render_height': 540,
    'render_width': 960,
    'seconds': 1.0,
    'camera_path': [
        {
            'camera_to_world': mat.flatten().tolist(),
            'fov': 55.0,
            'aspect': 16.0 / 9.0
        } for mat in poses
    ]
}
with open(camera_path_file, 'w', encoding='utf-8') as f:
    json.dump(path_dict, f, indent=2)
print(f'✅ [MEASURED] تم تجهيز مسار اللقطات الـ 6 في: {camera_path_file.name}')

# 4. رندر اللقطات لكل تجربة متاحة
variant_renders = {}
for var_id, var_info in experiment_outputs.items():
    cfg_p = var_info['config']
    var_render_dir = RENDER_DIR / var_id
    var_render_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n🎬 رندر المشاهد للتجربة {var_id}...')
    render_cmd = (
        f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render camera-path '
        f'--load-config {cfg_p} '
        f'--camera-path-filename {camera_path_file} '
        f'--output-path {var_render_dir} '
        f'--output-format images '
        f'--image-format png'
    )
    subprocess.run(render_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    rendered_imgs = sorted(list(var_render_dir.glob('**/*.png')))
    if not rendered_imgs:
        rendered_imgs = sorted(list(var_render_dir.parent.glob(f'{var_render_dir.name}/**/*.png')))
    variant_renders[var_id] = rendered_imgs
    print(f'   تم إنتاج {len(rendered_imgs)} إطاراً مصوراً.')

# 5. توليد لوحات المقارنة الأفقية (Contact Sheets)
contact_sheets = []
valid_vars = [vid for vid, imgs in variant_renders.items() if len(imgs) == 6]
if valid_vars:
    for p_idx in range(6):
        imgs_for_pose = [Image.open(variant_renders[vid][p_idx]) for vid in valid_vars]
        w, h = imgs_for_pose[0].size
        sheet_w = w * len(valid_vars)
        sheet_h = h + 50
        sheet = Image.new('RGB', (sheet_w, sheet_h), color=(25, 25, 25))
        draw = ImageDraw.Draw(sheet)

        for v_i, (vid, img) in enumerate(zip(valid_vars, imgs_for_pose)):
            sheet.paste(img, (v_i * w, 50))
            draw.text((v_i * w + 20, 15), f'[{vid}] {pose_labels[p_idx]}', fill=(255, 255, 255))

        sheet_path = RENDER_DIR / f'contact_sheet_pose_{p_idx+1:02d}.png'
        sheet.save(sheet_path)
        contact_sheets.append(sheet_path)
        print(f'   🖼️ [MEASURED] تم حفظ لوحة المقارنة: {sheet_path.name}')


## 📋 7. جدول المقارنة الشامل وتقييم الفرضيات الهندسية
تجميع كافة المقاييس الكمية واستطالة الجاوسيان في جدول موحد، يليه استنتاج منهجي واضح لما تؤكده أو تنفيه الأرقام.

In [ ]:
import math
import json
from pathlib import Path

print('=' * 80)
print('🏆 7. النتائج النهائية ومصفوفة تقييم الفرضيات (Maskan 3D Evaluation Matrix)')
print('=' * 80)

# 1. طباعة الجدول الشامل
headers = ['Variant', 'Iters', 'PSNR (dB)', 'SSIM', 'LPIPS', 'Gaussians', 'Size (MB)', 'Elong (>10)', 'Time (min)']
print(f'| {" | ".join(headers)} |')
print('|' + '|'.join([':---:' for _ in headers]) + '|')

summary_rows = []
for var in VARIANTS:
    vid = var['id']
    if vid not in eval_results:
        continue
    ev = eval_results[vid]
    el = elong_stats.get(vid, {})
    row = [
        vid,
        str(var['iters']),
        f"{ev['psnr']:.2f}" if not math.isnan(ev['psnr']) else 'N/A',
        f"{ev['ssim']:.4f}" if not math.isnan(ev['ssim']) else 'N/A',
        f"{ev['lpips']:.4f}" if not math.isnan(ev['lpips']) else 'N/A',
        f"{ev['gaussians']:,}",
        f"{ev['size_mb']:.2f}",
        f"{el.get('elong_share', 0.0):.2f}%",
        f"{ev['train_min']:.2f}"
    ]
    summary_rows.append(row)
    print(f'| {" | ".join(row)} |')

# حفظ الجدول بتنسيق CSV و JSON
with open(EXP_ROOT / 'experiments_summary.csv', 'w', encoding='utf-8') as f:
    f.write(','.join(headers) + '\n')
    for r in summary_rows:
        f.write(','.join(r) + '\n')

# 2. تقييم الفرضيات الأربع بلغة هندسية محايدة قائمة على الأرقام المقاسة
print('\n' + '=' * 80)
print('🔬 التقييم المنهجي للفرضيات الأربع بناءً على الأرقام المحسوبة:')
print('=' * 80)

# تقييم H1 (تمديد الخطوات والتنقية)
e0_el = elong_stats.get('E0', {}).get('elong_share', float('nan'))
e1_el = elong_stats.get('E1', {}).get('elong_share', float('nan'))
if not math.isnan(e0_el) and not math.isnan(e1_el):
    if e1_el < e0_el:
        print(f'1. الفرضية H1 (تأثير تمديد خطوات التنقية): مدعومة جزئياً بالأرقام [SUPPORTED].')
        print(f'   انخفضت نسبة الجاوسيان المتمدد من {e0_el:.2f}% في E0 إلى {e1_el:.2f}% في E1 بفضل 15 ألف خطوة تنقية بعد توقف الانقسام.')
    else:
        print(f'1. الفرضية H1: غير كافية بمفردها لإلغاء الاستطالة (E0: {e0_el:.2f}% مقابل E1: {e1_el:.2f}%).')
else:
    print('1. الفرضية H1: لا يمكن البت فيها نهائياً دون اكتمال تدريب E1.')

# تقييم H2 (تنظيم المقاييس)
e2_el = elong_stats.get('E2', {}).get('elong_share', float('nan'))
if not math.isnan(e2_el) and not math.isnan(e1_el):
    if e2_el < e1_el:
        print(f'2. الفرضية H2 (تنظيم المقاييس use_scale_regularization): مدعومة بقوة [SUPPORTED].')
        print(f'   تطبيق قيد max_gauss_ratio قلص نسبة الاستطالة من {e1_el:.2f}% في E1 إلى {e2_el:.2f}% في E2.')
    else:
        print(f'2. الفرضية H2: لم تظهر انخفاضاً ملحوظاً في نسبة الاستطالة (E2: {e2_el:.2f}%).')
else:
    print('2. الفرضية H2: لا يمكن البت فيها لعدم توفر ناتج E2.')

# تقييم H3 (الدقة الكاملة دون تخفيض)
e3_psnr = eval_results.get('E3', {}).get('psnr', float('nan'))
e1_psnr = eval_results.get('E1', {}).get('psnr', float('nan'))
if not math.isnan(e3_psnr) and not math.isnan(e1_psnr):
    diff = e3_psnr - e1_psnr
    print(f'3. الفرضية H3 (التدريب بالدقة الكاملة num_downscales=0): [MEASURED diff = {diff:+.2f} dB].')
    print(f'   فارق الـ PSNR بين التدريب بالدقة الكاملة والتدريب التدريجي هو {diff:+.2f} dB.')
else:
    print('3. الفرضية H3: لا يمكن البت فيها دون اكتمال E3.')

# تقييم H4 (درجة التوافقات الكروية)
e4_size = eval_results.get('E4', {}).get('size_mb', float('nan'))
e1_size = eval_results.get('E1', {}).get('size_mb', float('nan'))
if not math.isnan(e4_size) and not math.isnan(e1_size):
    print(f'4. الفرضية H4 (تأثير SH degree 3 على الحجم والانعكاسات): [MEASURED].')
    print(f'   تفعيل SH=3 رفع حجم الملف من {e1_size:.2f} MB إلى {e4_size:.2f} MB (زيادة بنسبة {e4_size/e1_size:.1f}x).')
else:
    print('4. الفرضية H4: لا يمكن البت فيها دون اكتمال E4.')


## 📦 8. حزم مخرجات التجارب والـ Checkpoint للتنزيل المباشر
تجميع كافة ملفات الـ PLY المصدّرة، وسجلات التدريب، وملفات الـ JSON، ولوحات المقارنة في ملف مضغوط واحد:
`/kaggle/working/exp_results.zip`.
كما يتم حزم نقطة حفظ دائمة للبيانات لتسهيل الاستئناف في أي وقت.

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

print('=' * 80)
print('📦 8. حزم مخرجات التجارب والـ Checkpoint الدائم')
print('=' * 80)

# 1. حزم أرشيف البيانات الدائم لتأمين الجلسة ضد إعادة التشغيل
DATASET_ZIP = Path('/kaggle/working/maskan3d_checkpoint.zip')
if not DATASET_ZIP.exists() and DATASET_DIR.exists():
    print(f'📦 جاري حزم مجلد البيانات في {DATASET_ZIP.name}...')
    with zipfile.ZipFile(DATASET_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as dz:
        for f in DATASET_DIR.rglob('*'):
            if f.is_file():
                dz.write(f, arcname=f'nerfstudio_dataset/{f.relative_to(DATASET_DIR)}')
    print(f'✅ [MEASURED] تم حفظ أرشيف البيانات: {DATASET_ZIP.name} ({DATASET_ZIP.stat().st_size / (1024**2):.2f} MB)')

# 2. حزم مخرجات التجارب الكاملة
ZIP_PATH = Path('/kaggle/working/exp_results.zip')
items_to_pack = [
    *list(EXP_ROOT.glob('**/*.ply')),
    *list(EXP_ROOT.glob('**/*.log')),
    *list(EXP_ROOT.glob('**/*.json')),
    *list(EXP_ROOT.glob('**/*.png')),
    *list(EXP_ROOT.glob('**/*.csv')),
    *list(EXP_ROOT.glob('**/config.yml')),
]

print(f'جاري ضغط {len(items_to_pack)} ملفاً في {ZIP_PATH.name}...')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in items_to_pack:
        if p.exists() and p.is_file():
            rel_name = p.relative_to(Path('/kaggle/working'))
            zf.write(p, arcname=str(rel_name))
            print(f'   + {rel_name} ({p.stat().st_size / (1024**2):.2f} MB) [MEASURED]')

zip_size = ZIP_PATH.stat().st_size / (1024**2)
print(f'\n🏆 [MEASURED] تم إنشاء حزمة النتائج بنجاح: {ZIP_PATH.name} ({zip_size:.2f} MB)')
print('=' * 80)
print('📥 اضغط على الرابط التالي لتنزيل الحزمة مباشرة إلى جهازك:')
display(FileLink(ZIP_PATH.name))
